# Integration of Periodic Splines
The spline being integrated is shown as a <span style="color:#1f77b4">**blue**</span> curve, with data samples at the integers represented with circles and stem lines. The sample at the origin, as well as its periodized replicates, is indicated by a red circle and stem line. The small black disks depict the knots of the spline. 

The lower integration bound is shown as a large <span style="color:#8c564b">**brown**</span> disk, while the upper integration bound is shown as a large <span style="color:#17becf">**cyan**</span> disk. In between, negative area contributions are shown in <span style="background-color:#eaf7f9;color:#7f7f7f">pale cyan</span> color, while positive area contributions are shown in <span style="background-color:#fbe6e5;color:#7f7f7f">pale red</span> color.

The curve shown in <span style="color:#000000">**black**</span> is computed as $\int_{a}^{x}\,f(t)\,{\mathrm{d}}t$, with $x\in[a,b].$ Finally, the area under the curve, computed as $\int_{a}^{b}\,f(t)\,{\mathrm{d}}t,$ is printed in text format.

In [ ]:
# Load the required libraries
from IPython.display import display
from IPython.display import Math
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 5 # Maximal spline degree
max_delay = 8.0 # Maximal absolute delay
min_bound = -5.0 # Minimal integration bound
max_bound = 20.0 # Maximal integration bound

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic splines with normal Gaussian coefficients
f = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 3)
f = f.plus(rng.choice([-1.0, 1.0]) * rng.uniform(0.05, 0.15) - f.mean())

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    a = 0.0,
    b = 9.0
):
    global f

    # Update of the spline
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = f.degree
        )
        # Slight departure from zero-mean
        f = f.plus(rng.choice([-1.0, 1.0]) * rng.uniform(0.05, 0.15) - f.mean())
    f.degree = degree
    f.delay = delay

    # Area under the curve
    auc = f.integrate(lower_bound = a, upper_bound = b)

    # Plot of the spline
    (fig, ax) = plt.subplots()
    f.plot(
        (fig, ax),
        plotdomain = sk.interval.ClosedOpen((min_bound, max_bound)),
        plotrange = sk.interval.Closed((-3.0, 3.0)),
        plotpoints = 200 + 1,
        curve_fmt = "-C0"
    )
    # Integration bounds
    (markerline, _, _) = plt.stem([a], [0.0], "C5o")
    markerline.set_markersize(11)
    (markerline, _, _) = plt.stem([b], [0.0], "C9o")
    markerline.set_markersize(11)
    # Integral
    x = np.linspace(a, b, 200 + 1)
    intgrl = np.array([f.integrate(lower_bound = a, upper_bound = x0) for x0 in x])
    ax.plot(x, intgrl, "-k")
    # Signed areas
    y = np.array([f.at(x0) for x0 in x])
    color = ["r", "C9"] if a < b else ["C9", "r"]
    ax.fill_between(
        x,
        y,
        0.0,
        where = (0.0 < y),
        color = color[0],
        alpha = 0.1,
        interpolate = True
    )
    ax.fill_between(
        x,
        y,
        0.0,
        where = (y < 0.0),
        color = color[1],
        alpha = 0.1,
        interpolate = True
    )
    plt.show()

    # Integral
    display(Math(
        r"\int_{{{:.2f}}}^{{{:.2f}}}\,f(x)\,{{\mathrm{{d}}}}x={}".format(a, b, auc)
    ))

# Interaction
widgets.interactive(
    update_plot,
    period = (1, max_period),
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    a = (min_bound, max_bound),
    b = (min_bound, max_bound)
)
